# Solution 4 — NLP formulation and diagnosis


In [ ]:
# Locate the repository so these editable modules work without installation.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Reload edited model modules before each cell during tutorial development.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

In [ ]:
import casadi as ca
import numpy as np
from casadi_tutorial.diagnostics import kkt_residuals

x = ca.MX.sym("x", 2)
f = (1 - x[0]) ** 2 + 100 * (x[1] - x[0] ** 2) ** 2
g = x[0] + x[1]
solver = ca.nlpsol(
    "exercise_solver",
    "ipopt",
    {"x": x, "f": f, "g": g},
    {"ipopt.print_level": 0, "print_time": False},
)
sol = solver(x0=[0, 1], lbx=-2, ubx=2, lbg=1, ubg=1)
xs = sol["x"]
gf = ca.Function("gf", [x], [ca.gradient(f, x)])
jg = ca.Function("jg", [x], [ca.jacobian(g, x)])
gfun = ca.Function("gfun", [x], [g])
res = kkt_residuals(gf(xs), jg(xs), sol["lam_g"], gfun(xs), [1], [1])
print(res)
assert solver.stats()["success"] and res["primal_inf"] < 1e-7